# 📐 Fundamentos: DBSCAN

## La idea, en una frase
> Un **cluster** es una región donde los puntos están **apretados**; lo que queda solo, en zonas vacías, es **ruido**.

En la sesión anterior K-Means nos obligó a decidir **cuántos** grupos queríamos y metió a **todos** los puntos en alguno, aunque no se pareciera a nadie. Además, sólo encuentra grupos **redondos** (cada punto va al centroide más cercano).

**DBSCAN** (*Density-Based Spatial Clustering of Applications with Noise*, Ester et al., 1996) cambia la pregunta: en lugar de "¿a qué centro estás más cerca?" pregunta **"¿cuántos vecinos tienes cerca?"**. Con eso:

| | K-Means | DBSCAN |
|---|---|---|
| ¿Hay que dar el número de clusters? | Sí (`n_clusters`) | **No**: sale solo |
| Forma de los clusters | Redondos (convexos) | **Cualquier forma** |
| Puntos raros (outliers) | Los mete a un cluster | Los marca como **ruido** (`-1`) |
| Parámetros | `n_clusters` | `eps` (radio) y `min_samples` (vecinos mínimos) |
| Punto débil | Formas raras, outliers | Clusters con **densidades muy distintas** |

<p align="center">
  <img src="img/DBSCAN_points.png" alt="Tipos de puntos en DBSCAN: núcleo, frontera y ruido" width="620" />
</p>

### Contenido
1. Densidad y ε-vecindad: núcleo, frontera y ruido
2. El algoritmo paso a paso
3. Los dos parámetros y su efecto
4. Cómo elegir `eps`: la gráfica de k-distancia
5. DBSCAN vs K-Means: lunas y círculos
6. Densidades variables: donde DBSCAN falla y HDBSCAN lo resuelve
7. Escalado, distancias y estructuras de búsqueda
8. ⚠️ Errores comunes

---
# 0️⃣ Preparación

Ejecuta las siguientes celdas **una sola vez**.

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_clusters(df, x, y, labels, centroides=None)` | Puntos en 2D coloreados por cluster (el ruido en gris) |
| `plot_clusters_3d(df, x, y, z, labels)` | Lo mismo en 3D (se puede rotar) |
| `plot_codo(ks, inercias)` | Método del codo para elegir k |
| `plot_metricas_k(tabla)` | Silhouette, Davies-Bouldin y Calinski-Harabasz según k |
| `plot_silueta(X, labels)` | Coeficiente de silueta de cada punto, por cluster |
| `plot_k_distancia(X, k, eps=None)` | Distancia al k-ésimo vecino: ayuda a elegir `eps` de DBSCAN |
| `plot_varianza_explicada(pca)` | Varianza explicada por componente y acumulada |
| `plot_cargas(pca, columnas)` | Peso de cada variable original en cada componente |
| `plot_biplot(scores, pca, columnas, labels=None)` | Puntos en PC1-PC2 con una flecha por variable |

In [ ]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _etiquetas_texto(labels):
    """Convierte etiquetas de cluster a texto para que plotly use colores discretos; -1 = 'Ruido'."""
    labels = np.asarray(labels)
    return np.where(labels == -1, 'Ruido', np.char.add('Cluster ', labels.astype(str)))


def _orden_clusters(texto):
    """Orden de la leyenda: Cluster 0, 1, 2, … y 'Ruido' al final."""
    unicos = sorted(set(texto), key=lambda t: (t == 'Ruido', int(t.split()[-1]) if t != 'Ruido' else 0))
    return {'cluster': unicos}


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas'
    )
    fig.update_layout(width=1000, height=1000, title_font_size=20)
    fig.update_traces(diagonal_visible=True, marker={'size': 4, 'opacity': 0.6})
    fig.show()


def plot_clusters(df, x, y, labels, centroides=None, title='Clusters'):
    """Dispersión 2D coloreada por cluster. El ruido de DBSCAN (-1) aparece en gris.
    centroides: arreglo (k, 2) en las mismas unidades que x, y (opcional)."""
    datos = df[[x, y]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter(
        datos,
        x=x,
        y=y,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    if centroides is not None:
        centroides = np.asarray(centroides)
        fig.add_trace(go.Scatter(
            x=centroides[:, 0],
            y=centroides[:, 1],
            mode='markers',
            marker={'symbol': 'x', 'size': 16, 'color': 'black', 'line': {'width': 2}},
            name='Centroides'
        ))
    fig.update_layout(width=800, height=550)
    fig.show()


def plot_clusters_3d(df, x, y, z, labels, title='Clusters en 3D'):
    """Dispersión 3D coloreada por cluster (se puede rotar con el mouse)."""
    datos = df[[x, y, z]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter_3d(
        datos,
        x=x,
        y=y,
        z=z,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    fig.update_traces(marker={'size': 4})
    fig.update_layout(width=850, height=650)
    fig.show()


def plot_codo(ks, inercias):
    """Método del codo: inercia (suma de distancias² al centroide) para cada k."""
    fig = px.line(
        x=list(ks),
        y=list(inercias),
        markers=True,
        labels={'x': 'Número de clusters (k)', 'y': 'Inercia'},
        title='Método del codo'
    )
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_metricas_k(tabla):
    """Silhouette, Davies-Bouldin y Calinski-Harabasz para cada k (tabla de tabla_metricas_k)."""
    metricas = [
        ('silhouette', 'Silhouette (↑ mejor)'),
        ('davies_bouldin', 'Davies-Bouldin (↓ mejor)'),
        ('calinski_harabasz', 'Calinski-Harabasz (↑ mejor)'),
    ]
    fig = make_subplots(rows=1, cols=3, subplot_titles=[t for _, t in metricas])
    for i, (col, _) in enumerate(metricas, start=1):
        fig.add_trace(go.Scatter(x=tabla['k'], y=tabla[col], mode='lines+markers', name=col), row=1, col=i)
        fig.update_xaxes(title_text='k', dtick=1, row=1, col=i)
    fig.update_layout(width=1150, height=400, showlegend=False, title='Métricas internas según k')
    fig.show()


def plot_silueta(X, labels, title=None):
    """Gráfica de silueta: el ancho de cada barra es el coeficiente de un punto, agrupado por cluster.
    La línea punteada es el promedio. Los puntos de ruido (-1) se excluyen."""
    from sklearn.metrics import silhouette_samples, silhouette_score

    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    X, labels = X[mask], labels[mask]
    valores = silhouette_samples(X, labels)
    promedio = silhouette_score(X, labels)

    fig = go.Figure()
    y_inf = 10
    for c in np.unique(labels):
        v = np.sort(valores[labels == c])
        y = np.arange(y_inf, y_inf + len(v))
        fig.add_trace(go.Scatter(
            x=v, y=y, fill='tozerox', mode='lines', line={'width': 0.5}, name=f'Cluster {c}'
        ))
        y_inf += len(v) + 10
    fig.add_vline(x=promedio, line_dash='dash', line_color='red',
                  annotation_text=f'promedio = {promedio:.3f}', annotation_position='top')
    fig.update_layout(
        title=title or f'Gráfica de silueta (k = {len(np.unique(labels))})',
        xaxis_title='Coeficiente de silueta',
        yaxis={'showticklabels': False, 'title': 'Puntos (ordenados por cluster)'},
        width=750,
        height=500
    )
    fig.show()


def plot_k_distancia(X, k, eps=None):
    """Distancia que necesita cada punto para tener k vecinos (contándose a sí mismo), ordenada.
    Usa k = min_samples: igual que DBSCAN, el punto cuenta como uno de sus k vecinos, así que un punto
    es núcleo con un eps dado si su distancia en esta curva es <= eps. El 'codo' sugiere un buen eps.
    eps (opcional) dibuja una línea horizontal de referencia."""
    from sklearn.neighbors import NearestNeighbors

    # n_neighbors=k incluye al propio punto (distancia 0), como min_samples en DBSCAN
    vecinos = NearestNeighbors(n_neighbors=k).fit(X)
    distancias, _ = vecinos.kneighbors(X)
    d_k = np.sort(distancias[:, -1])

    fig = px.line(
        x=np.arange(len(d_k)),
        y=d_k,
        labels={'x': 'Puntos ordenados por distancia', 'y': f'Distancia para tener {k} vecinos (incluido él)'},
        title=f'Gráfica de k-distancia (k = {k})'
    )
    if eps is not None:
        fig.add_hline(y=eps, line_dash='dash', line_color='red',
                      annotation_text=f'eps = {eps}', annotation_position='top left')
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_varianza_explicada(pca):
    """Scree plot: varianza explicada por componente (barras) y acumulada (línea)."""
    ratio = pca.explained_variance_ratio_
    n = np.arange(1, len(ratio) + 1)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=n, y=ratio, name='Por componente'))
    fig.add_trace(go.Scatter(x=n, y=np.cumsum(ratio), mode='lines+markers', name='Acumulada'))
    for umbral in (0.80, 0.95):
        fig.add_hline(y=umbral, line_dash='dot', line_color='gray',
                      annotation_text=f'{umbral:.0%}', annotation_position='right')
    fig.update_layout(
        title='Varianza explicada por componente',
        xaxis={'title': 'Componente principal', 'dtick': 1 if len(ratio) <= 30 else None},
        yaxis={'title': 'Proporción de varianza', 'tickformat': '.0%', 'range': [0, 1.05]},
        width=800,
        height=450
    )
    fig.show()


def plot_cargas(pca, columnas, n_componentes=None):
    """Mapa de calor de las cargas (loadings): cuánto pesa cada variable original en cada componente."""
    n = n_componentes or pca.n_components_
    cargas = pd.DataFrame(
        pca.components_[:n].T,
        index=columnas,
        columns=[f'PC{i + 1}' for i in range(n)]
    ).round(2)
    fig = px.imshow(
        cargas,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        aspect='auto',
        title='Cargas de cada variable en los componentes'
    )
    fig.update_layout(width=200 + 90 * n, height=150 + 40 * len(columnas))
    fig.show()


def plot_biplot(scores, pca, columnas, labels=None, escala=None, title='Biplot PC1 vs PC2'):
    """Puntos en el plano PC1-PC2 más una flecha por variable original (sus cargas).
    scores: salida de pca.transform(X). labels (opcional) colorea los puntos por cluster o grupo."""
    scores = np.asarray(scores)
    datos = pd.DataFrame({'PC1': scores[:, 0], 'PC2': scores[:, 1]})
    if labels is not None:
        datos['grupo'] = _etiquetas_texto(labels) if np.issubdtype(np.asarray(labels).dtype, np.integer) \
            else np.asarray(labels).astype(str)
    fig = px.scatter(datos, x='PC1', y='PC2', color='grupo' if labels is not None else None,
                     opacity=0.6, title=title)
    cargas = pca.components_[:2].T
    escala = escala or 0.9 * np.abs(scores[:, :2]).max() / np.abs(cargas).max()
    for (cx, cy), nombre in zip(cargas * escala, columnas):
        fig.add_annotation(x=cx, y=cy, ax=0, ay=0, xref='x', yref='y', axref='x', ayref='y',
                           showarrow=True, arrowhead=2, arrowwidth=1.5, arrowcolor='black')
        fig.add_annotation(x=cx * 1.08, y=cy * 1.08, text=nombre, showarrow=False, font={'size': 12})
    r = pca.explained_variance_ratio_
    fig.update_layout(
        xaxis_title=f'PC1 ({r[0]:.1%} de la varianza)',
        yaxis_title=f'PC2 ({r[1]:.1%} de la varianza)',
        width=850,
        height=650
    )
    fig.show()

### 🧰 Funciones de clustering

Hacen en una línea lo que de otra forma repetirías con cada modelo:

| Función | ¿Para qué sirve? |
|---|---|
| `evaluar_clustering(nombre, X, labels)` | Número de clusters, % de ruido, silhouette, Davies-Bouldin y Calinski-Harabasz (sin contar el ruido) |
| `tabla_metricas_k(X, ks)` | Ajusta K-Means para cada k y regresa inercia y métricas en una tabla |
| `perfil_clusters(df, labels, columnas)` | Tamaño y promedio de cada variable por cluster |

In [ ]:
# Funciones de clustering
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score


def evaluar_clustering(nombre, X, labels):
    """Métricas internas de un agrupamiento. El ruido (-1) se excluye de las métricas,
    pero se reporta qué porcentaje de los puntos quedó como ruido."""
    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    n_clusters = len(np.unique(labels[mask]))
    fila = {
        'modelo': nombre,
        'n_clusters': n_clusters,
        'pct_ruido': float(round(100 * (~mask).mean(), 1)),
    }
    if n_clusters >= 2:
        fila['silhouette'] = float(round(silhouette_score(X[mask], labels[mask]), 3))
        fila['davies_bouldin'] = float(round(davies_bouldin_score(X[mask], labels[mask]), 3))
        fila['calinski_harabasz'] = float(round(calinski_harabasz_score(X[mask], labels[mask]), 1))
    else:
        fila.update({'silhouette': np.nan, 'davies_bouldin': np.nan, 'calinski_harabasz': np.nan})
    return fila


def tabla_metricas_k(X, ks, random_state=42):
    """Ajusta K-Means para cada k y regresa una tabla con inercia, silhouette, Davies-Bouldin y Calinski-Harabasz."""
    filas = []
    for k in ks:
        modelo = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(X)
        fila = evaluar_clustering(f'k={k}', X, modelo.labels_)
        filas.append({'k': k, 'inercia': float(round(modelo.inertia_, 1)), **{
            m: fila[m] for m in ('silhouette', 'davies_bouldin', 'calinski_harabasz')}})
    return pd.DataFrame(filas)


def perfil_clusters(df, labels, columnas):
    """Promedio de cada variable por cluster, con tamaño y porcentaje. Sirve para ponerle nombre a cada grupo."""
    datos = df[columnas].copy()
    datos['cluster'] = np.asarray(labels)
    perfil = datos.groupby('cluster')[columnas].mean().round(1)
    perfil.insert(0, 'n', datos['cluster'].value_counts().sort_index())
    perfil.insert(1, 'pct', (100 * perfil['n'] / perfil['n'].sum()).round(1))
    perfil.index = ['Ruido' if c == -1 else f'Cluster {c}' for c in perfil.index]
    return perfil

### 🎨 Funciones de este notebook
Para **ver** los círculos de radio ε usamos matplotlib (plotly no dibuja círculos con escala real tan fácil). No necesitas entender el código de dibujo: sólo llama a las funciones.

| Función | ¿Para qué sirve? |
|---|---|
| `tipos_de_punto(modelo)` | Dice si cada punto es `núcleo`, `frontera` o `ruido` (de un DBSCAN ya ajustado) |
| `dibujar_dbscan(X, modelo, circulos=None)` | Núcleos grandes, fronteras pequeñas y ruido como ✕; `circulos='frontera'` o `'todos'` dibuja el radio ε |
| `grilla_dbscan(X, eps_values, min_samples_values)` | Una gráfica por combinación de parámetros |

In [ ]:
# Funciones de este notebook (dibujos con matplotlib)
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from sklearn.cluster import DBSCAN, HDBSCAN, KMeans

COLORES = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#17becf']


def tipos_de_punto(modelo):
    """'núcleo', 'frontera' o 'ruido' para cada punto de un DBSCAN ya ajustado."""
    tipos = np.full(len(modelo.labels_), 'frontera', dtype=object)
    tipos[modelo.core_sample_indices_] = 'núcleo'
    tipos[modelo.labels_ == -1] = 'ruido'
    return tipos


def dibujar_dbscan(X, modelo, ax=None, circulos=None, titulo=None, etiquetas=False):
    """Núcleos = puntos grandes, frontera = puntos chicos, ruido = ✕ gris.
    circulos: None, 'frontera' (círculo de radio eps alrededor de cada punto frontera) o 'todos'."""
    X = np.asarray(X)
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 5.5))
    labels = modelo.labels_
    tipos = tipos_de_punto(modelo)
    eps = modelo.eps
    for i, (x, y) in enumerate(X):
        color = 'gray' if labels[i] == -1 else COLORES[labels[i] % len(COLORES)]
        if circulos == 'todos' or (circulos == 'frontera' and tipos[i] == 'frontera'):
            ax.add_patch(Circle((x, y), eps, fill=False, linestyle='--', linewidth=0.8, color=color, alpha=0.7))
    for tipo, marcador, tam in [('núcleo', 'o', 70), ('frontera', 'o', 25), ('ruido', 'x', 40)]:
        m = tipos == tipo
        colores = ['gray' if l == -1 else COLORES[l % len(COLORES)] for l in labels[m]]
        ax.scatter(X[m, 0], X[m, 1], s=tam, c=colores, marker=marcador,
                   edgecolors='black' if tipo != 'ruido' else None, linewidths=0.5, label=tipo)
    if etiquetas:
        for i, (x, y) in enumerate(X):
            ax.annotate(str(i), (x, y), textcoords='offset points', xytext=(6, 6), fontsize=9)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    ax.set_title(titulo or f'eps = {eps}, min_samples = {modelo.min_samples} → '
                 f'{n_clusters} cluster{"" if n_clusters == 1 else "s"}, {(labels == -1).mean():.0%} ruido',
                 fontsize=10)
    ax.set_aspect('equal')
    return ax


def grilla_dbscan(X, eps_values, min_samples_values):
    """Ajusta DBSCAN para cada combinación y dibuja una gráfica por combinación."""
    fig, axes = plt.subplots(len(min_samples_values), len(eps_values),
                             figsize=(4 * len(eps_values), 3.3 * len(min_samples_values)))
    for i, ms in enumerate(min_samples_values):
        for j, eps in enumerate(eps_values):
            modelo = DBSCAN(eps=eps, min_samples=ms).fit(X)
            dibujar_dbscan(X, modelo, ax=axes[i, j])
            axes[i, j].set_xticks([])
            axes[i, j].set_yticks([])
    fig.tight_layout()
    plt.show()

---
# 1️⃣ Densidad y ε-vecindad: núcleo, frontera y ruido

Dos ingredientes:
- **ε (`eps`)**: un radio. La **ε-vecindad** de un punto $p$ son todos los puntos a distancia $\le \varepsilon$ de él: $N_\varepsilon(p) = \{q : d(p, q) \le \varepsilon\}$
- **`min_samples`** (en las slides, *MinPts*): cuántos puntos tiene que haber en esa vecindad para decir que la zona es **densa**. ⚠️ En scikit-learn **el propio punto cuenta**.

Con eso, cada punto cae en uno de tres tipos:

| Tipo | Regla | Papel |
|---|---|---|
| 🔵 **Núcleo** (*core*) | Tiene $\ge$ `min_samples` puntos en su ε-vecindad | "Siembra" y hace crecer el cluster |
| 🔹 **Frontera** (*border*) | No es núcleo, pero está dentro de la ε-vecindad de algún núcleo | Pertenece al cluster, pero no lo expande |
| ✕ **Ruido** (*noise*) | Ni núcleo ni cerca de un núcleo | Etiqueta `-1` |

Y dos ideas para "pegar" puntos:
- **Densamente alcanzable**: $q$ se alcanza desde $p$ si hay una **cadena de núcleos** $p \to p_1 \to \dots \to q$, cada uno a menos de ε del anterior.
- **Densamente conectados**: $p$ y $q$ están conectados si ambos se alcanzan desde un mismo núcleo. **Un cluster = todos los puntos densamente conectados entre sí** (por eso el cluster "crece" siguiendo la densidad, sin importar su forma).

Veámoslo con **13 puntos** a mano, con `eps = 0.8` y `min_samples = 4`:

In [ ]:
# 13 puntos en 2D (ya en la misma escala)
puntos = np.array([
    [1.0, 1.0], [1.5, 1.2], [1.2, 1.7], [1.8, 1.6], [2.2, 1.3], [1.4, 0.6], [2.9, 1.5],
    [3.6, 1.9], [5.0, 4.0], [5.4, 4.3], [5.1, 4.7], [5.7, 4.1], [0.2, 3.8]
])

from sklearn.metrics import pairwise_distances

eps_toy = 0.8
distancias = pairwise_distances(puntos)              # matriz 13 × 13 de distancias euclidianas
vecinos = (distancias <= eps_toy).sum(axis=1)        # cuántos puntos hay en su ε-vecindad (incluido él mismo)

pd.DataFrame({'x': puntos[:, 0], 'y': puntos[:, 1], 'vecinos_en_eps': vecinos})

In [ ]:
dbscan_toy = DBSCAN(eps=eps_toy, min_samples=4).fit(puntos)

pd.DataFrame({
    'vecinos_en_eps': vecinos,
    'tipo': tipos_de_punto(dbscan_toy),
    'cluster': dbscan_toy.labels_
})

In [ ]:
dibujar_dbscan(puntos, dbscan_toy, circulos='todos', etiquetas=True)
plt.show()

✅ **Qué observar:**
- Los puntos **0, 1, 2, 3 y 4** tienen 4 o más vecinos en su círculo (contándose a sí mismos): son **núcleos** y forman el cluster 0.
- El **5** (3 vecinos) y el **6** (2 vecinos) no son núcleos, pero caen dentro del círculo de un núcleo: son **frontera** y se quedan en el cluster 0.
- El **7** sólo se tiene a sí mismo y el único punto a menos de 0.8 (el 6) **no es núcleo**: es **ruido**. Estar cerca de un punto frontera **no basta**.
- Arriba a la derecha, el **8** y el **9** son núcleos y arrastran al **10** y al **11** como frontera → cluster 1. El **12** está solo → ruido.
- Nadie le dijo a DBSCAN que había **2 clusters**: salió de la densidad.

### ✍️ Mini-ejercicio 1
¿Qué pasa si exigimos **más densidad**? Ajusta un DBSCAN con `eps = 0.8` y `min_samples = 5` sobre `puntos`, guárdalo en `dbscan_toy_5` y dibújalo.

In [ ]:
# 1. dbscan_toy_5 = DBSCAN(eps=..., min_samples=...).fit(puntos)
# 2. dibujar_dbscan(puntos, dbscan_toy_5, circulos='todos', etiquetas=True) y plt.show()
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Cuántos núcleos quedan? ¿Qué le pasó al grupo de arriba a la derecha? ¿Por qué?

RELLENAR CON TUS COMENTARIOS

---
# 2️⃣ El algoritmo paso a paso

1. Para cada punto, cuenta sus vecinos en el radio ε → marca los **núcleos**.
2. Toma un núcleo que todavía no tenga cluster y crea un cluster **nuevo** con él.
3. **Expande**: agrega a todos sus vecinos; si un vecino también es núcleo, agrega a **sus** vecinos, y así sucesivamente (la cadena de núcleos).
4. Cuando ya no se puede crecer, vuelve al paso 2 con otro núcleo libre.
5. Lo que nunca se alcanzó desde un núcleo es **ruido** (`-1`).

💡 Diferencias con K-Means:
- **No hay centroides** ni iteraciones que "mejoren": se recorre cada punto una vez.
- Es **determinista** en núcleos y ruido. Sólo un punto frontera que toca a dos clusters puede quedar en uno u otro según el orden de los datos.
- No hay `predict` para puntos nuevos: DBSCAN describe **estos** datos.

---
# 3️⃣ Los dos parámetros y su efecto

Usamos las famosas **dos lunas** (`make_moons`): dos grupos curvos que K-Means no puede separar. Primero las escalamos (DBSCAN mide distancias).

Mira primero un DBSCAN "bien calibrado" sobre unas lunas con 8 puntos raros agregados, con el círculo ε dibujado alrededor de cada punto **frontera**:

In [ ]:
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler

X_lunas_raw, _ = make_moons(n_samples=200, noise=0.1, random_state=42)
raros = np.random.default_rng(42).uniform([-1.5, -1.0], [2.5, 1.5], size=(8, 2))   # 8 puntos sueltos
X_lunas_ruido = StandardScaler().fit_transform(np.vstack([X_lunas_raw, raros]))

dbscan_lunas_ruido = DBSCAN(eps=0.3, min_samples=6).fit(X_lunas_ruido)
print(pd.Series(tipos_de_punto(dbscan_lunas_ruido)).value_counts())

dibujar_dbscan(X_lunas_ruido, dbscan_lunas_ruido, circulos='frontera')
plt.show()

✅ **Qué observar:** **176 núcleos** forman el "esqueleto" de cada luna, **22 puntos frontera** quedan en las orillas (su círculo toca a un núcleo) y **10 puntos** quedan como ruido: casi todos son los sueltos que agregamos (los que caen pegados a una luna se integran a ella como frontera).

### ¿Y si movemos los parámetros?
Ahora una grilla de 3 × 3 sobre 300 lunas: columnas = `eps`, filas = `min_samples`.

| Parámetro | Muy chico | Muy grande |
|---|---|---|
| `eps` | Casi nadie tiene vecinos → **todo es ruido** o muchos clusters diminutos | Todo queda conectado → **un solo cluster** |
| `min_samples` | Cualquier par de puntos forma un cluster (muy sensible al ruido) | Sólo las zonas muy densas sobreviven → **mucho ruido** |

In [ ]:
X_lunas_raw, y_lunas = make_moons(n_samples=300, noise=0.08, random_state=42)
X_lunas = StandardScaler().fit_transform(X_lunas_raw)

grilla_dbscan(
    X_lunas,
    eps_values=[0.1, 0.3, 0.5],
    min_samples_values=[3, 10, 20]
)

✅ **Qué observar:**
- **`eps = 0.1`** (columna izquierda): el radio es tan chico que salen 30 clusters diminutos con `min_samples = 3`, y con 10 o 20 **todo** es ruido.
- **`eps = 0.5`** (derecha): el radio alcanza para "saltar" de una luna a la otra y todo se vuelve **1 cluster**… salvo con `min_samples = 20`, donde exigir más densidad vuelve a separarlas.
- **`eps = 0.3`** con `min_samples` 3 o 10: las **2 lunas** perfectas. Con 20, la exigencia es demasiada para ese radio: 54% de ruido.
- 💡 Los dos parámetros **se compensan**: más radio permite pedir más vecinos. Por eso se eligen **juntos**.

---
# 4️⃣ Cómo elegir `eps`: la gráfica de k-distancia

Probar a ciegas es caro. La receta práctica (Ester et al. y Schubert et al., 2017):

1. **Fija `min_samples` primero** con la regla práctica: $\text{min\_samples} \ge \text{dimensiones} + 1$; un buen arranque es $2 \times \text{dimensiones}$. Con datos ruidosos o muchos puntos, súbelo.
2. Para cada punto calcula el radio que necesita para juntar **$k$ puntos contándose a sí mismo**, con $k$ = `min_samples` (igual que cuenta DBSCAN). Ese radio es el `eps` mínimo con el que ese punto sería núcleo.
3. Ordénalas de menor a mayor y grafícalas. Los puntos dentro de un cluster tienen distancias **chicas y parecidas**; los de ruido, **grandes**. El **codo** (donde la curva se dispara) es un buen candidato para `eps`.

Nuestras lunas tienen 2 dimensiones → `min_samples` ≈ 4 o 5. Usamos $k = 5$:

In [ ]:
plot_k_distancia(X_lunas, k=5, eps=0.3)

✅ **Qué observar:** la curva es plana y baja (la mitad de los puntos junta sus 5 puntos en un radio menor a **0.14**) y se levanta al final: el 95% está por debajo de **0.25**. La línea roja en **0.3** cae justo después del codo, y es el `eps` que separó bien las lunas en la grilla.

> 💡 El codo es un **punto de partida**, no una verdad: en datos reales la curva puede no tener un codo claro. Después se afina probando valores cercanos y midiendo (lo harás en el laboratorio).

### ✍️ Mini-ejercicio 2
Si quisieras exigir más densidad, con `min_samples = 10`:
1. Dibuja la gráfica de k-distancia de `X_lunas` con `k = 10`.
2. Elige un `eps` mirando el codo y ajusta `dbscan_lunas_10 = DBSCAN(eps=<tu eps>, min_samples=10).fit(X_lunas)`.
3. Revisa cuántos clusters y cuánto ruido salen con `evaluar_clustering('DBSCAN k=10', X_lunas, dbscan_lunas_10.labels_)`.

In [ ]:
# 1. plot_k_distancia(X_lunas, k=10)
# 2. dbscan_lunas_10 = DBSCAN(eps=..., min_samples=10).fit(X_lunas)
# 3. evaluar_clustering('DBSCAN k=10', X_lunas, dbscan_lunas_10.labels_)
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿El codo quedó más arriba o más abajo que con $k = 5$? ¿Por qué tiene sentido? ¿Cuántos clusters y qué % de ruido obtuviste con tu `eps`?

RELLENAR CON TUS COMENTARIOS

---
# 5️⃣ DBSCAN vs K-Means: lunas y círculos

Aquí sí conocemos los grupos reales (`y_lunas`), así que podemos usar el **ARI** (*Adjusted Rand Index*): 1 = idéntico a la realidad, 0 = como asignar al azar. Recuerda que en la práctica **no** tenemos esa etiqueta.

In [ ]:
from sklearn.metrics import adjusted_rand_score

kmeans_lunas = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_lunas)
dbscan_lunas = DBSCAN(eps=0.3, min_samples=5).fit(X_lunas)

plot_clusters(pd.DataFrame(X_lunas, columns=['x1', 'x2']), 'x1', 'x2', kmeans_lunas.labels_,
              centroides=kmeans_lunas.cluster_centers_, title='K-Means (k=2) en las lunas')
plot_clusters(pd.DataFrame(X_lunas, columns=['x1', 'x2']), 'x1', 'x2', dbscan_lunas.labels_,
              title='DBSCAN (eps=0.3, min_samples=5) en las lunas')

In [ ]:
comparacion_lunas = [
    {**evaluar_clustering('K-Means (k=2)', X_lunas, kmeans_lunas.labels_),
     'ARI': round(adjusted_rand_score(y_lunas, kmeans_lunas.labels_), 3)},
    {**evaluar_clustering('DBSCAN', X_lunas, dbscan_lunas.labels_),
     'ARI': round(adjusted_rand_score(y_lunas, dbscan_lunas.labels_), 3)},
]
pd.DataFrame(comparacion_lunas)

✅ **Qué observar:**
- K-Means corta las lunas con una **línea recta** (cada punto va al centroide más cercano): ARI = **0.479**. DBSCAN sigue la curva y recupera las dos lunas **exactas**: ARI = **1.0**.
- 🤔 Pero la **silhouette** dice lo contrario: K-Means **0.494** vs DBSCAN **0.38**. La silhouette premia grupos **redondos y separados**; dos lunas abrazadas no lo son. **Una métrica interna alta no garantiza grupos correctos**: siempre mira la gráfica.

Ahora dos **círculos concéntricos**, el caso extremo:

In [ ]:
from sklearn.datasets import make_circles

X_circ_raw, y_circ = make_circles(n_samples=500, factor=0.5, noise=0.05, random_state=42)
X_circ = StandardScaler().fit_transform(X_circ_raw)

kmeans_circ = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_circ)
dbscan_circ = DBSCAN(eps=0.3, min_samples=5).fit(X_circ)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].scatter(X_circ[:, 0], X_circ[:, 1], c=[COLORES[l] for l in kmeans_circ.labels_], s=15)
axes[0].set_title(f'K-Means: ARI = {adjusted_rand_score(y_circ, kmeans_circ.labels_):.3f}')
dibujar_dbscan(X_circ, dbscan_circ, ax=axes[1],
               titulo=f'DBSCAN: ARI = {adjusted_rand_score(y_circ, dbscan_circ.labels_):.3f}')
for ax in axes:
    ax.set_aspect('equal')
plt.show()

✅ **Qué observar:** K-Means parte los anillos por la mitad (ARI ≈ **0**: igual que al azar). DBSCAN encuentra el anillo de adentro y el de afuera (ARI = **1.0**) porque cada anillo es una cadena continua de núcleos.

---
# 6️⃣ Densidades variables: donde DBSCAN falla y HDBSCAN lo resuelve

DBSCAN usa **un solo** `eps` para todo el mapa. ¿Qué pasa si hay grupos **muy apretados** y otros **muy dispersos**? Generamos 3 grupos de 250 puntos: dos muy densos y cercanos, y uno grande y disperso.

In [ ]:
from sklearn.datasets import make_blobs

X_dens_raw, y_dens = make_blobs(
    n_samples=[250, 250, 250],
    centers=[[0, 0], [1.6, 0], [6, 3]],
    cluster_std=[0.25, 0.25, 1.5],
    random_state=42
)
X_dens = StandardScaler().fit_transform(X_dens_raw)

dbscan_eps_chico = DBSCAN(eps=0.1, min_samples=10).fit(X_dens)
dbscan_eps_grande = DBSCAN(eps=0.3, min_samples=10).fit(X_dens)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
dibujar_dbscan(X_dens, dbscan_eps_chico, ax=axes[0])
dibujar_dbscan(X_dens, dbscan_eps_grande, ax=axes[1])
plt.show()

🚩 **No hay un `eps` que funcione:**
- Con **`eps = 0.1`** los dos grupos densos salen bien, pero el grupo disperso **desaparece**: sus 250 puntos son ruido (34% del total).
- Con **`eps = 0.3`** el grupo disperso aparece, pero los dos densos se **fusionan** en uno de 500 puntos.

### HDBSCAN: DBSCAN para todos los `eps` a la vez
**HDBSCAN** (*Hierarchical* DBSCAN, Campello et al., 2013; en scikit-learn desde la versión 1.3 como `sklearn.cluster.HDBSCAN`) hace, en esencia, esto:
1. Corre DBSCAN para **todos** los radios posibles, de grande a chico, y arma un **árbol**: con radio grande todo es un cluster; al achicarlo, los clusters se parten o se "evaporan" en ruido.
2. Se queda con los clusters **más estables**: los que sobreviven en un rango amplio de radios.

Así, cada grupo se detecta **con el radio que le queda**. Ya no hay `eps`: el parámetro principal es **`min_cluster_size`**, el tamaño mínimo que tú consideras un grupo (mucho más fácil de explicar a negocio).

In [ ]:
hdbscan_dens = HDBSCAN(min_cluster_size=10, copy=True).fit(X_dens)

comparacion_dens = []
for nombre, labels in [('DBSCAN eps=0.1', dbscan_eps_chico.labels_),
                       ('DBSCAN eps=0.3', dbscan_eps_grande.labels_),
                       ('HDBSCAN min_cluster_size=10', hdbscan_dens.labels_)]:
    comparacion_dens.append({**evaluar_clustering(nombre, X_dens, labels),
                             'ARI': round(adjusted_rand_score(y_dens, labels), 3)})
pd.DataFrame(comparacion_dens)

In [ ]:
plot_clusters(pd.DataFrame(X_dens, columns=['x1', 'x2']), 'x1', 'x2', hdbscan_dens.labels_,
              title='HDBSCAN (min_cluster_size=10)')

✅ **Qué observar:** HDBSCAN encuentra los **3 grupos** con sólo **2.3%** de ruido y ARI = **0.966**, sin que le digamos ningún radio. DBSCAN tenía que escoger entre perder el grupo disperso (34.1% de ruido) o fusionar los densos (ARI = 0.476).

🤔 ¿Y el ARI de **0.976** de `eps = 0.1`? Es engañoso: el ARI trata el `-1` como un grupo más, y aquí el "ruido" coincide casi exacto con el grupo disperso. Pero en la práctica **ruido no es un segmento**: DBSCAN te dijo que esos 250 puntos no pertenecen a nada. Con HDBSCAN sí forman un grupo.

> ⚠️ `copy=True` sólo evita un aviso de scikit-learn 1.9 (en la 1.10 cambia el default); no afecta el resultado.

### ✍️ Mini-ejercicio 3
¿HDBSCAN también resuelve las lunas sin elegir `eps`? Ajusta `hdbscan_lunas = HDBSCAN(min_cluster_size=<tu valor>, copy=True).fit(X_lunas)` con un tamaño mínimo razonable (las lunas tienen 150 puntos cada una) y calcula su ARI contra `y_lunas`. Prueba al menos dos valores.

In [ ]:
# 1. hdbscan_lunas = HDBSCAN(min_cluster_size=..., copy=True).fit(X_lunas)
# 2. evaluar_clustering('HDBSCAN', X_lunas, hdbscan_lunas.labels_)
# 3. adjusted_rand_score(y_lunas, hdbscan_lunas.labels_)
# 4. Repite con otro min_cluster_size
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Cuántos clusters encontró con cada `min_cluster_size`? ¿Qué tan sensible es el resultado a ese parámetro comparado con lo que viste con `eps` en la grilla?

RELLENAR CON TUS COMENTARIOS

---
# 7️⃣ Escalado, distancias y estructuras de búsqueda

### Escalar antes de medir distancias
`eps` es una distancia. Si una variable está en **pesos** y otra en **años**, la de pesos domina: una diferencia de \$1,000 pesa 1,000 veces más que un año. Repetimos las lunas, pero con la segunda variable multiplicada por 1,000 (como si estuviera en otra unidad):

In [ ]:
X_lunas_sin_escalar = X_lunas_raw * np.array([1, 1000])     # misma forma, otra unidad en la 2ª variable

for nombre, X_ in [('Sin escalar', X_lunas_sin_escalar),
                   ('Con StandardScaler', StandardScaler().fit_transform(X_lunas_sin_escalar))]:
    labels_ = DBSCAN(eps=0.3, min_samples=5).fit_predict(X_)
    print(f'{nombre:20} → {evaluar_clustering(nombre, X_, labels_)}')

✅ **Qué observar:** sin escalar, con el mismo `eps = 0.3` **todo** es ruido (en la escala de miles, nadie tiene vecinos a 0.3). Después de `StandardScaler` vuelven las **2 lunas** sin ruido. **Escala siempre** antes de DBSCAN (salvo coordenadas GPS con Haversine, que ya están en la misma unidad: ángulos).

### Distancias que acepta DBSCAN (`metric=`)

| Distancia | Fórmula | Cuándo | En scikit-learn |
|---|---|---|---|
| Euclidiana | $\sqrt{\sum_i (p_i - q_i)^2}$ | Default; variables continuas escaladas | `metric='euclidean'` |
| Manhattan (L1) | $\sum_i \lvert p_i - q_i \rvert$ | Muchas dimensiones; menos sensible a una variable extrema | `metric='manhattan'` |
| Haversine | $2R \arcsin\sqrt{\sin^2\frac{\Delta\varphi}{2} + \cos\varphi_1\cos\varphi_2\sin^2\frac{\Delta\lambda}{2}}$ | Coordenadas GPS (latitud $\varphi$, longitud $\lambda$) | `metric='haversine'`, datos **en radianes** `[lat, lon]`, `eps = km / 6371` |

### ¿Cómo encuentra los vecinos tan rápido? (`algorithm=`)
Comparar cada punto con todos cuesta $n^2$ distancias: con 1 millón de puntos son $10^{12}$. Las estructuras de árbol descartan regiones enteras de un golpe:

| Estructura | Idea | Bueno para | Métricas |
|---|---|---|---|
| **KD-Tree** | Parte el espacio con cortes alternando ejes (como un árbol de decisión sobre las coordenadas) | Pocas dimensiones (< ~20) | Euclidiana, Manhattan… |
| **Ball-Tree** | Agrupa los puntos en **esferas anidadas** | Más dimensiones o métricas especiales | Casi cualquiera, incluida **Haversine** |
| **Brute** | Calcula todas las distancias | Datos chicos o muy alta dimensión | Cualquiera |

Por defecto `algorithm='auto'` elige solo; no necesitas tocarlo.

---
# ⚠️ Errores comunes

### 1. Calcular la silhouette con el ruido incluido
`silhouette_score(X, labels)` trata al `-1` como **un cluster más**: un "cluster" de puntos sueltos que no se parecen entre sí. Usa `evaluar_clustering`, que lo excluye y reporta aparte el % de ruido.

In [ ]:
from sklearn.metrics import silhouette_score

labels_ruidosos = DBSCAN(eps=0.2, min_samples=10).fit_predict(X_lunas)
sin_ruido = labels_ruidosos != -1

print(f'% de ruido:                 {100 * (~sin_ruido).mean():.1f}%')
print(f'Silhouette con el ruido:    {silhouette_score(X_lunas, labels_ruidosos):.3f}   ← mezcla peras con manzanas')
print(f'Silhouette sin el ruido:    {silhouette_score(X_lunas[sin_ruido], labels_ruidosos[sin_ruido]):.3f}')

✅ Con el ruido incluido la silhouette es **0.136**; sin él, **0.507**. Pero ojo con la trampa contraria: **mandar más puntos a ruido infla la silhouette** (sólo se evalúan los puntos "fáciles"). Por eso **siempre** se reporta la silhouette **junto con** el % de ruido: aquí 36.7% de los puntos quedaron fuera.

### 2. Elegir `eps` sin escalar
`eps` está en las unidades de los datos. Si cambias de escala (o se te olvida escalar), el mismo `eps` significa otra cosa (sección 7).

### 3. Haversine con grados
`metric='haversine'` espera **radianes** y regresa la distancia **en radianes** (un ángulo sobre una esfera de radio 1). Para pasar a km se multiplica por el radio de la Tierra (6,371 km):

In [ ]:
from sklearn.metrics.pairwise import haversine_distances

macroplaza = [25.6697, -100.3097]      # [latitud, longitud] en grados
valle_oriente = [25.6450, -100.3500]

bien = haversine_distances(np.radians([macroplaza]), np.radians([valle_oriente]))[0, 0] * 6371
mal = haversine_distances([macroplaza], [valle_oriente])[0, 0] * 6371

print(f'Con radianes: {bien:.2f} km   ✅')
print(f'Con grados:   {mal:,.0f} km   ❌ (¡interpreta los grados como radianes!)')

✅ De la Macroplaza a Valle Oriente hay unos **4.9 km**; con grados sale un número absurdo. Y el `eps` también va en radianes: **`eps = km / 6371`** (1 km → `eps = 1/6371`).

### 4. Otros tropiezos
- **`min_samples` muy chico** (1 o 2): cada par de puntos es un "cluster". Usa al menos dimensiones + 1.
- **Contar el `-1` como un cluster** al decir "DBSCAN encontró k grupos".
- **Comparar la silhouette de K-Means (todos los puntos) con la de DBSCAN (sin ruido)** como si fuera la misma prueba: se calcularon sobre **conjuntos distintos**.
- **Esperar `predict`**: DBSCAN no asigna puntos nuevos; vuelve a ajustarlo (o usa el cluster del núcleo más cercano).

---
## 📝 Resumen
1. DBSCAN agrupa por **densidad**: núcleos (≥ `min_samples` vecinos en ε), frontera (cerca de un núcleo) y ruido.
2. No pide el número de clusters, encuentra **formas arbitrarias** y **separa el ruido**.
3. Elige `min_samples` primero (≥ dimensiones + 1) y luego `eps` con la **gráfica de k-distancia**; afina probando valores cercanos.
4. Falla con **densidades muy distintas**: ahí **HDBSCAN** prueba todos los radios y sólo pide `min_cluster_size`.
5. **Escala** antes de medir distancias; con GPS usa **Haversine en radianes** y `eps = km / 6371`.
6. Reporta la silhouette **sin ruido** y **siempre** junto con el % de ruido.